# ML Assignment 2: Dry Bean Classification
Complete learning notebook with two switchable data sources, EDA, leakage-safe preprocessing, five models, six metrics, export, and verification.

## 1. Setup and paths

In [1]:
from pathlib import Path
import sys, pandas as pd, numpy as np
ROOT=Path.cwd().parent if Path.cwd().name=='notebooks' else Path.cwd()
sys.path.insert(0,str(ROOT))
from config import *
from src.data_loader import load_from_uci,load_from_kaggle_file,validate_dataset
from src.preprocessing import clean_dataset,split_dataset
from src.train_models import build_models,train_and_save
from src.evaluate_models import evaluate_all
from src.visualization import save_eda,save_comparison,save_confusion
print(ROOT)

D:\ML\ML_Assignment_2


## 2. Dataset loading: enable exactly one option

In [2]:
# OPTION A: UCI (ACTIVE)
df = load_from_uci(RAW_DATA_PATH)

# OPTION B: KAGGLE (COMMENTED)
# KAGGLE_FILE = ROOT / "data" / "raw" / "Dry_Bean_Dataset.xlsx"
# df = load_from_kaggle_file(KAGGLE_FILE, target_column="Class")

print(validate_dataset(df))
assert df.shape[1]==17 and TARGET_COLUMN in df.columns

{'rows': 13611, 'columns': 17, 'features': 16, 'target': 'Class', 'missing': 0, 'duplicates': 68, 'classes': {'DERMASON': 3546, 'SIRA': 2636, 'SEKER': 2027, 'HOROZ': 1928, 'CALI': 1630, 'BARBUNYA': 1322, 'BOMBAY': 522}, 'dtypes': {'Area': 'int64', 'Perimeter': 'float64', 'MajorAxisLength': 'float64', 'MinorAxisLength': 'float64', 'AspectRation': 'float64', 'Eccentricity': 'float64', 'ConvexArea': 'int64', 'EquivDiameter': 'float64', 'Extent': 'float64', 'Solidity': 'float64', 'roundness': 'float64', 'Compactness': 'float64', 'ShapeFactor1': 'float64', 'ShapeFactor2': 'float64', 'ShapeFactor3': 'float64', 'ShapeFactor4': 'float64', 'Class': 'object'}}


UCI source: https://archive.ics.uci.edu/dataset/602/dry+bean+dataset (ID 602). Kaggle manual alternative: https://www.kaggle.com/datasets/zeynepazl/dry-beans/data. Both loaders normalize the same 16-feature plus `Class` schema.

## 3. EDA and validation

In [3]:
display(df.head()); display(df.tail()); print(df.shape); print(df.columns.tolist()); df.info(); display(df.describe(include='all').T); print(df.isna().sum()); print('Duplicates:',df.duplicated().sum()); display(df['Class'].value_counts().to_frame('count')); display((df['Class'].value_counts(normalize=True)*100).round(2).to_frame('percent')); save_eda(df,TARGET_COLUMN,FIGURE_DIR)

,Area,Perimeter,MajorAxisLength,MinorAxisLength,AspectRation,Eccentricity,ConvexArea,EquivDiameter,Extent,Solidity,roundness,Compactness,ShapeFactor1,ShapeFactor2,ShapeFactor3,ShapeFactor4,Class
0,28395,610.291,208.178117,173.888747,1.197191,0.549812,28715,190.141097,0.763923,0.988856,0.958027,0.913358,0.007332,0.003147,0.834222,0.998724,SEKER
1,28734,638.018,200.524796,182.734419,1.097356,0.411785,29172,191.272751,0.783968,0.984986,0.887034,0.953861,0.006979,0.003564,0.909851,0.998430,SEKER
2,29380,624.110,212.826130,175.931143,1.209713,0.562727,29690,193.410904,0.778113,0.989559,0.947849,0.908774,0.007244,0.003048,0.825871,0.999066,SEKER
3,30008,645.884,210.557999,182.516516,1.153638,0.498616,30724,195.467062,0.782681,0.976696,0.903936,0.928329,0.007017,0.003215,0.861794,0.994199,SEKER
4,30140,620.134,201.847882,190.279279,1.060798,0.333680,30417,195.896503,0.773098,0.990893,0.984877,0.970516,0.006697,0.003665,0.941900,0.999166,SEKER


,Area,Perimeter,MajorAxisLength,MinorAxisLength,AspectRation,Eccentricity,ConvexArea,EquivDiameter,Extent,Solidity,roundness,Compactness,ShapeFactor1,ShapeFactor2,ShapeFactor3,ShapeFactor4,Class
13606,42097,759.696,288.721612,185.944705,1.552728,0.765002,42508,231.515799,0.714574,0.990331,0.916603,0.801865,0.006858,0.001749,0.642988,0.998385,DERMASON
13607,42101,757.499,281.576392,190.713136,1.476439,0.735702,42494,231.526798,0.799943,0.990752,0.922015,0.822252,0.006688,0.001886,0.676099,0.998219,DERMASON
13608,42139,759.321,281.539928,191.187979,1.472582,0.734065,42569,231.631261,0.729932,0.989899,0.918424,0.822730,0.006681,0.001888,0.676884,0.996767,DERMASON
13609,42147,763.779,283.382636,190.275731,1.489326,0.741055,42667,231.653247,0.705389,0.987813,0.907906,0.817457,0.006724,0.001852,0.668237,0.995222,DERMASON
13610,42159,772.237,295.142741,182.204716,1.619841,0.786693,42600,231.686223,0.788962,0.989648,0.888380,0.784997,0.007001,0.001640,0.616221,0.998180,DERMASON


(13611, 17)
['Area', 'Perimeter', 'MajorAxisLength', 'MinorAxisLength', 'AspectRation', 'Eccentricity', 'ConvexArea', 'EquivDiameter', 'Extent', 'Solidity', 'roundness', 'Compactness', 'ShapeFactor1', 'ShapeFactor2', 'ShapeFactor3', 'ShapeFactor4', 'Class']
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13611 entries, 0 to 13610
Data columns (total 17 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Area             13611 non-null  int64  
 1   Perimeter        13611 non-null  float64
 2   MajorAxisLength  13611 non-null  float64
 3   MinorAxisLength  13611 non-null  float64
 4   AspectRation     13611 non-null  float64
 5   Eccentricity     13611 non-null  float64
 6   ConvexArea       13611 non-null  int64  
 7   EquivDiameter    13611 non-null  float64
 8   Extent           13611 non-null  float64
 9   Solidity         13611 non-null  float64
 10  roundness        13611 non-null  float64
 11  Compactness      13611 non-nul

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Area,13611.0,NaN,NaN,NaN,53048.284549,29324.095717,20420.0,36328.0,44652.0,61332.0,254616.0
Perimeter,13611.0,NaN,NaN,NaN,855.283459,214.289696,524.736,703.5235,794.941,977.213,1985.37
MajorAxisLength,13611.0,NaN,NaN,NaN,320.141867,85.694186,183.601165,253.303633,296.883367,376.495012,738.860154
MinorAxisLength,13611.0,NaN,NaN,NaN,202.270714,44.970091,122.512653,175.84817,192.431733,217.031741,460.198497
AspectRation,13611.0,NaN,NaN,NaN,1.583242,0.246678,1.024868,1.432307,1.551124,1.707109,2.430306
Eccentricity,13611.0,NaN,NaN,NaN,0.750895,0.092002,0.218951,0.715928,0.764441,0.810466,0.911423
ConvexArea,13611.0,NaN,NaN,NaN,53768.200206,29774.915817,20684.0,36714.5,45178.0,62294.0,263261.0
EquivDiameter,13611.0,NaN,NaN,NaN,253.06422,59.17712,161.243764,215.068003,238.438026,279.446467,569.374358
Extent,13611.0,NaN,NaN,NaN,0.749733,0.049086,0.555315,0.718634,0.759859,0.786851,0.866195
Solidity,13611.0,NaN,NaN,NaN,0.987143,0.00466,0.919246,0.98567,0.988283,0.990013,0.994677


Area               0
Perimeter          0
MajorAxisLength    0
MinorAxisLength    0
AspectRation       0
Eccentricity       0
ConvexArea         0
EquivDiameter      0
Extent             0
Solidity           0
roundness          0
Compactness        0
ShapeFactor1       0
ShapeFactor2       0
ShapeFactor3       0
ShapeFactor4       0
Class              0
dtype: int64
Duplicates: 68


,count
Class,
DERMASON,3546
SIRA,2636
SEKER,2027
HOROZ,1928
CALI,1630
BARBUNYA,1322
BOMBAY,522


,percent
Class,
DERMASON,26.05
SIRA,19.37
SEKER,14.89
HOROZ,14.17
CALI,11.98
BARBUNYA,9.71
BOMBAY,3.84


## 4. Cleaning and fixed split
Duplicates are dropped only after they are measured. The split is stratified. Pipeline transformations are fit only on training data, preventing leakage.

In [4]:
df_clean=clean_dataset(df); df_clean.to_csv(PROCESSED_DATA_PATH,index=False)
X_train,X_test,y_train,y_test=split_dataset(df_clean,TARGET_COLUMN,TEST_SIZE,RANDOM_STATE)
test_data=X_test.copy(); test_data[TARGET_COLUMN]=y_test.values; test_data.to_csv(TEST_DATA_PATH,index=False)
print(X_train.shape,X_test.shape,y_train.value_counts(normalize=True).round(3).to_dict())

(10834, 16) (2709, 16) {'DERMASON': 0.262, 'SIRA': 0.195, 'SEKER': 0.15, 'HOROZ': 0.137, 'CALI': 0.12, 'BARBUNYA': 0.098, 'BOMBAY': 0.039}


## 5. Algorithms
Logistic Regression provides a linear probabilistic baseline. Decision Tree learns interpretable nonlinear rules but can overfit. KNN predicts by nearby scaled samples and can be slow at inference. GaussianNB is fast but assumes conditionally independent Gaussian features. RBF SVM finds nonlinear maximum-margin boundaries and often performs strongly but is computationally heavier.

In [5]:
models=build_models(RANDOM_STATE)
models=train_and_save(models,X_train,y_train,MODEL_DIR)
list(models)

['logistic_regression', 'decision_tree', 'knn', 'naive_bayes', 'svm']

## 6. Consistent multiclass evaluation

In [6]:
results,details=evaluate_all(models,X_test,y_test,REPORT_DIR)
save_comparison(results,FIGURE_DIR)
for name,model in models.items(): save_confusion(details[name][2],model.classes_,name,FIGURE_DIR)
display(results.style.format({c:'{:.4f}' for c in ['Accuracy','AUC','Precision','Recall','F1','MCC']}))

,ML Model Name,Accuracy,AUC,Precision,Recall,F1,MCC
4,svm,0.9240,0.9944,0.9359,0.9346,0.9352,0.9080
0,logistic_regression,0.9195,0.9947,0.9311,0.9303,0.9306,0.9028
2,knn,0.9136,0.9854,0.9268,0.9237,0.9250,0.8956
1,decision_tree,0.9029,0.9596,0.9138,0.9148,0.9140,0.8826
3,naive_bayes,0.8970,0.9911,0.9081,0.9082,0.9072,0.8762


## 7. Evidence-based observations

In [7]:
winner=results.iloc[0]
print('Overall winner by ordered F1, MCC, AUC:',winner['ML Model Name'])
print('Highest AUC:',results.loc[results['AUC'].idxmax(),'ML Model Name'])
print('Best macro F1:',results.loc[results['F1'].idxmax(),'ML Model Name'])
print('Best MCC:',results.loc[results['MCC'].idxmax(),'ML Model Name'])

Overall winner by ordered F1, MCC, AUC: svm
Highest AUC: logistic_regression
Best macro F1: svm
Best MCC: svm


## 8. Saved-model verification

In [8]:
import joblib
for name in models:
 loaded=joblib.load(MODEL_DIR/f'{name}.joblib')
 assert np.array_equal(loaded.predict(X_test.head(20)),models[name].predict(X_test.head(20)))
print('All five serialized pipelines verified.')

All five serialized pipelines verified.
